# COMP0005 - GROUP COURSEWORK
# Experimental Evaluation of Search Data Structures and Algorithms

The cell below defines **AbstractSearchInterface**, an interface to support basic insert/search operations; you will need to implement this three times, to realise your three search data structures of choice among: (1) *2-3 Tree*, (2) *AVL Tree*, (3) *LLRB BST*; (4) *B-Tree*; and (5) *Scapegoat Tree*. <br><br>**Do NOT modify the next cell** - use the dedicated cells further below for your implementation instead. <br>

In [ ]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

Use the cell below to define any auxiliary data structure and python function you may need. Leave the implementation of the main API to the next code cells instead.

In [ ]:
# ADD AUXILIARY DATA STRUCTURE DEFINITIONS AND HELPER CODE HERE

class AvlNode:
    def __init__(self, key):
        self.key: str = key
        self.height: int = 0
        self.left: AvlNode | None = None
        self.right: AvlNode | None = None


class AvlHelper:
    
    @staticmethod
    def heightOf(root: AvlNode | None) -> int:
        # height of root = edges from root to furthest leaf
        # -1 because height(leaf) = 1 + max(-1,-1) = 0
        if root is None:
            return -1
        else:
            return 1 + max(
                root.left.height if root.left is not None else -1,
                root.right.height if root.right is not None else -1
            )


    @staticmethod
    def balanceOf(root: AvlNode) -> int:
        return AvlHelper.heightOf(root.left) - AvlHelper.heightOf(root.right)


    @staticmethod
    def rotateRight(root: AvlNode) -> AvlNode:
        # returns the new root of the subtree (root.left)
        assert root.left is not None, "Right rotation requires root.left to be a Node"
        
        # perform rotation
        left = root.left
        leftRight = root.left.right
        root.left = leftRight
        left.right = root

        #update heights
        root.height = AvlHelper.heightOf(root)
        left.height = AvlHelper.heightOf(left)

        return left
    

    @staticmethod
    def rotateLeft(root: AvlNode) -> AvlNode:
        # returns the new root of the subtree (root.right)
        assert root.right is not None, "Left rotation requires root.right to be a Node"

        # perform rotation
        right = root.right
        rightLeft = root.right.left
        root.right = rightLeft
        right.left = root

        # update heights
        root.height = AvlHelper.heightOf(root)
        right.height = AvlHelper.heightOf(right)

        return right


    @staticmethod
    def searchElement(element: str, root: AvlNode | None) -> bool:
        # returns whether the element is in root's subtree

        if root == None:
            return False
        elif root.key == element:
            return True
        elif root.key > element:
            return AvlHelper.searchElement(element, root.left)
        else:
            return AvlHelper.searchElement(element, root.right)
    

    @staticmethod
    def insertElement(element: str, root: AvlNode | None) -> AvlNode:
        # insert into the right/left subtree
        if root is None:
            return AvlNode(element)
        elif root.key > element:
            root.left = AvlHelper.insertElement(element, root.left)
        elif root.key < element:
            root.right = AvlHelper.insertElement(element, root.right)

        # root.left and root.right are now balanced
        # height of child nodes may have changed, so height of root may have changed
        root.height = AvlHelper.heightOf(root)        

        balance = AvlHelper.balanceOf(root)

        # case 1: left subtree too heavy and imbalance from left's left subtree
        if balance > 1 and root.left is not None and element < root.left.key:
            return AvlHelper.rotateRight(root)
        # case 2: left subtree too heavy and imbalance from left's right subtree
        elif balance > 1 and root.left is not None and element > root.left.key:
            root.left = AvlHelper.rotateLeft(root.left)
            return AvlHelper.rotateRight(root)
        # case 3: mirror case 1
        if balance < -1 and root.right is not None and element > root.right.key:
            return AvlHelper.rotateLeft(root)
        # case 4: mirror case 2
        elif balance < -1 and root.right is not None and element < root.right.key:
            root.right = AvlHelper.rotateRight(root.right)
            return AvlHelper.rotateLeft(root)
        # else balanced
        else:
            return root
    
    
    @staticmethod
    def printBfs(root: AvlNode) -> None:
        # for testing
        q: list[AvlNode | None] = [root]
        while len(q) > 0:
            node = q.pop(0)
            if node:
                print(node.key)
                q.append(node.left)
                q.append(node.right)

class LLRBNode:
    def __init__(self, key):
        self.key: str = key
        self.left: LLRBNode | None = None
        self.right: LLRBNode | None = None
        self.color: bool = True # True: red, False: black

class LLRBHelper:

    @staticmethod
    def rotate_left(root: LLRBNode) -> LLRBNode:
        right = root.right
        root.right = right.left
        right.left = root

        right.color = root.color
        root.color = True

        return right

    @staticmethod
    def rotate_right(root: LLRBNode) -> LLRBNode:
        left = root.left
        root.left = left.right
        left.right = root

        left.color = root.color
        root.color = True

        return left

    @staticmethod
    def flip_colors(node: LLRBNode) -> LLRBNode:
        node.color = not node.color
        node.left.color = not node.left.color
        node.right.color = not node.right.color
        return node

    @staticmethod
    def is_red(node: LLRBNode | None) -> bool:
        return node is not None and node.color

    @staticmethod
    def search(key: str, root: LLRBNode | None) -> bool:
        while root:
            if key < root.key:
                root = root.left
            elif key > root.key:
                root = root.right
            else:
                return True
        return False

    @staticmethod
    def insert(key: str, root: LLRBNode | None) -> LLRBNode:
        if not root:
            return LLRBNode(key)
        if key < root.key:
            root.left = LLRBHelper.insert(key, root.left)
        elif key > root.key:
            root.right = LLRBHelper.insert(key, root.right)
        else:
            return root
        if LLRBHelper.is_red(root.right) and not LLRBHelper.is_red(root.left):
            root = LLRBHelper.rotate_left(root)
        if LLRBHelper.is_red(root.left) and LLRBHelper.is_red(root.left.left):
            root = LLRBHelper.rotate_right(root)
        if LLRBHelper.is_red(root.left) and LLRBHelper.is_red(root.right):
            root = LLRBHelper.flip_colors(root)

        return root


Use the cell below to implement the requested API by means of **2-3 Tree** (if among your chosen data structure).

In [ ]:
class TwoThreeTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found    

Use the cell below to implement the requested API by means of **AVL Tree** (if among your chosen data structure).

In [ ]:
class AVLTree(AbstractSearchInterface):

    def __init__(self, root):
        self.root = root


    def insertElement(self, element):
        inserted = False 
        # ADD YOUR CODE HERE

        canInsert = not AvlHelper.searchElement(element, self.root)
        if canInsert:
            self.root = AvlHelper.insertElement(element, self.root)
        inserted = canInsert
        return inserted
    
    
    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE
        
        found = AvlHelper.searchElement(element, self.root)
        return found

In [ ]:
# tmp test block

r = AvlNode("a")
t = AVLTree(r)
for i in range(98, 98+14):
  t.insertElement(chr(i))
AvlHelper.printBfs(t.root)

Use the cell below to implement the requested API by means of **LLRB BST** (if among your chosen data structure).

In [ ]:
class LLRBBST(AbstractSearchInterface):

    def __init__(self, root=None):
        self.root = root
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
        if self.searchElement(element):
            return False
        self.root = LLRBHelper.insert(element, self.root)
        self.root.color = False
        inserted = True
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE
        found = LLRBHelper.search(element, self.root)
        
        return found

In [ ]:
# Test
def print_tree(node, indent="", side="root"):
    if node is None:
        print(f"{indent}[{side}] None")
        return

    color = "R" if node.color else "B"
    print(f"{indent}[{side}] {node.key}({color})")
    print_tree(node.left, indent + "   ", "L")
    print_tree(node.right, indent + "   ", "R")

tree = LLRBBST()

print(tree.insertElement("10"))
print(tree.insertElement("20"))
print(tree.insertElement("30"))
print(tree.insertElement("20"))

print(tree.searchElement("10"))
print(tree.searchElement("20"))
print(tree.searchElement("30"))
print(tree.searchElement("40"))

print("root color is black:", tree.root.color == False)

print_tree(tree.root)

Use the cell below to implement the requested API by means of **B-Tree** (if among your chosen data structure).

In [ ]:
class BTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found

Use the cell below to implement the requested API by means of **Scapegoat Tree** (if among your chosen data structure).

In [ ]:
class ScapegoatTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found 

Use the cell below to implement the **synthetic data generator** needed by your experimental framework (be mindful of code readability and reusability).

In [ ]:
import string
import random

class TestDataGenerator():
    '''
    A class to represent a synthetic data generator.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
    
    #ADD YOUR CODE HERE
    
    def __init__():
        pass
    

Use the cell below to implement the requested **experimental framework** (be mindful of code readability and reusability).

In [ ]:
import timeit
import matplotlib

class ExperimentalFramework():
    '''
    A class to represent an experimental framework.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
            
    #ADD YOUR CODE HERE
    
    def __init__():
        pass
    

Use the cell below to illustrate the python code you used to **fully evaluate** your three chosen search data structures and algortihms. The code below should illustrate, for example, how you made used of the **TestDataGenerator** class to generate test data of various size and properties; how you instatiated the **ExperimentalFramework** class to  evaluate each data structure using such data, collect information about their execution time, plot results, etc. Any results you illustrate in the companion PDF report should have been generated using the code below.

In [ ]:
# ADD YOUR TEST CODE HERE 



